# ZH - Mesterséges intelligencia - Dealer detector

### Szabályok

- A rendelkezésre álló idő: **85 perc**, **+5 perc** feltöltés
- Egyéni munka, mindenki önállóan dolgozik (AI és egyéb humán segítség nélkül)
- A korábbi órák anyagai használhatóak a laborvezetővel egyeztetett módon
- A munkafüzetben megjelölt mezőkben dolgozzon (`#TODO`), de új cellákat is felvehet, igény szerint

__Beadás__ (http://zh.nik.lan): A kitöltött, elmentett (!) notebook, futási eredményekkel
- A _warning_-ok figyelmen kívül hagyhatóak

### Feladat: 
Készítsen osztályozó modellt, amely képes:
- személygépjárművek eladási adatai alapján megjósolni, hogy az autót magánszemély vagy kereskedő árulja-e (célváltozó: `seller_type`)
- __hipotézisünk__: a kereskedők drágábban árulják az azonos kategóriájú (évjárat, köbcenti, futott kilóméter, stb.) autókat

### 0. LÉPÉS: Alapvető könyvtárak importálása

(További könyvtárak importálására is szükség lehet.)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

### 1. LÉPÉS: Adat betöltése

- Töltse be az autók adatait tartalmazó `Car details v3.csv` fájlt egy `df` elnevezésű DataFrame objektumba
- Jelenítse meg az első három sort


In [2]:
# TODO
df = pd.read_csv('Car details v3.csv')
df.head(3)

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0


### 2. LÉPÉS: Adatok áttekintése

Jelenítse meg, hogy: 
- hány sor és hány oszlop található a DataFrame-ben
- az egyes oszlopoknak mi a típusa
- a szám típusú oszlopoknak mik az értéktartományai (minimum, maximum érték)
- a `seller_type` kategorikus oszlopnak mi az értékkészlete

In [3]:
# TODO
print(df.shape)
df.info()
print(df.describe())
print(df['seller_type'].unique())

(8128, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           8128 non-null   object 
 1   year           8128 non-null   int64  
 2   selling_price  8128 non-null   int64  
 3   km_driven      8128 non-null   int64  
 4   fuel           8128 non-null   object 
 5   seller_type    8128 non-null   object 
 6   transmission   8128 non-null   object 
 7   owner          8128 non-null   object 
 8   mileage        7907 non-null   object 
 9   engine         7907 non-null   object 
 10  max_power      7913 non-null   object 
 11  torque         7906 non-null   object 
 12  seats          7907 non-null   float64
dtypes: float64(1), int64(3), object(9)
memory usage: 825.6+ KB
              year  selling_price     km_driven        seats
count  8128.000000   8.128000e+03  8.128000e+03  7907.000000
mean   2013.804011   6.382718e+05  6

### 3. LÉPÉS: Adattisztítás

- jelenítse meg, hogy az melyik oszlop hány üres (NaN) cellát tartalmaz
- minden olyan sort távolítson el, amelynek valamely attribútuma (cellája) üres
- az eltávolított sorok számát tárolja el egy removed_records változóban, amit írjon ki


In [4]:
# TODO

print(df.isna().sum())

prev_rows = df.shape[0]
df = df.dropna()

removed_records = prev_rows - df.shape[0]
print("removed:", removed_records)

name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          221
engine           221
max_power        215
torque           222
seats            221
dtype: int64
removed: 222


### 4. LÉPÉS: Feature engineering

- a `transmission` oszolopot képezze le 1 (kézi váltós) és 2 (automata) értékekre
- az `engine` oszlopot alakítsa mértékegység (`CC`) nélküli, numerikus adattá
- az `owner` oszlop képezze le számokra: 0: teszt autó, 1, ..., 4: tulajdonosok száma (4: négy vagy több)
- a `seller_type` oszlop értéke legyen 0: ha magánszemély az eladó, különben 1 
- a `fuel` oszlopot bontsa fel 1-hot-enconding alapján 4 oszlopra (CNG, Diesel, ...)
- törölje a `name`, `mileage`, `engine`, `max_power`, `torque` oszlopokat
- ellenőrizze, hogy most már minden oszlop numerikus típusú-e

In [5]:
# TODO

# hipotezis: year, selling_price, km_driven, fuel, transmission, owner, seats, => price, seller

df['transmission'] = df['transmission'].apply(lambda x: 2 if x == 'Automatic' else 1) 

# engine, "1248 CC"
df['engine'] = df['engine'].str.replace("CC", "")
df['engine'] = df['engine'].astype(float)

mapping = {
    'First Owner': 1,
    'Second Owner': 2,
    'Third Owner': 3,
    'Fourth & Above Owner': 4,
    'Test Drive Car': 0
}
df['owner'] = df['owner'].map(mapping)

df['seller_type'].unique() # ['Individual', 'Dealer', 'Trustmark Dealer']
mapping = {
    'Individual': 0,
    'Dealer': 1,
    'Trustmark Dealer': 1
}
df['seller_type'] = df['seller_type'].map(mapping)

df = pd.get_dummies(df, columns=['fuel']) # fuel oszlop helyett 4

df = df.drop(columns=['name', 'mileage', 'engine', 'max_power', 'torque'])

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7906 entries, 0 to 8127
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   year           7906 non-null   int64  
 1   selling_price  7906 non-null   int64  
 2   km_driven      7906 non-null   int64  
 3   seller_type    7906 non-null   int64  
 4   transmission   7906 non-null   int64  
 5   owner          7906 non-null   int64  
 6   seats          7906 non-null   float64
 7   fuel_CNG       7906 non-null   bool   
 8   fuel_Diesel    7906 non-null   bool   
 9   fuel_LPG       7906 non-null   bool   
 10  fuel_Petrol    7906 non-null   bool   
dtypes: bool(4), float64(1), int64(6)
memory usage: 525.0 KB


### 5. LÉPÉS: Modell inputok, outputok előkészítése

- válassza le az adatokat a célváltozó (`seller_type`) halmazáról: előbbi legyen `X`, az utóbbi `y` elnevezésű
- ossza fel az `X` és `y` halmazokat tanító és teszt részhalmazra 80%-20% arányban
- ezekre használja a`X_train`, `y_train`, `X_test`, `y_test` változóneveket
- állítson be egy `random_state` értéket, a kísérlet megismételhetősége érdekében
- használja a `stratify=y` opciót, hogy a tanító és teszthalmazokban arányosan legyenek különböző kimenetelek

In [6]:
# TODO
from sklearn.model_selection import train_test_split

X = df.drop(columns=["seller_type"])
y = df["seller_type"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 6. LÉPÉS: Modell tanítása

- hozzon létre egy __Döntési fa__ osztályozó modellt
- a fa maximális mélysége 20 legyen
- használja a __model1__ változónevet ehhez a modellhez
- állítson be egy random_state értéket, a kísérlet megismételhetősége érdekében
- tanítsa be

In [7]:
# TODO
from sklearn.tree import DecisionTreeClassifier
model1 = DecisionTreeClassifier(max_depth=20, random_state=42)
model1.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=20, random_state=42)

### 7. LÉPÉS: Modell kiértékelése

- értékelje ki a modellt a __teszt__ halmazon a következő metrikákkal:
    - pontosság (accuracy)
    - precizitás (precision)
    - szenzitivitás (recall)
    - igazságmátrix (confusion matrix)

In [8]:
# TODO
y_pred = model1.predict(X_test)
print(classification_report(y_pred, y_test))
print(confusion_matrix(y_pred, y_test))

              precision    recall  f1-score   support

           0       0.93      0.93      0.93      1312
           1       0.68      0.68      0.68       270

    accuracy                           0.89      1582
   macro avg       0.81      0.81      0.81      1582
weighted avg       0.89      0.89      0.89      1582

[[1226   86]
 [  87  183]]


### 8. LÉPÉS: Második modell

- válasszon és tanítson be ugyanezen adatokon egy másik, **lineáris osztályozó modellt** (ne használjon SVM-et, a hosszú futási idő miatt, warning esetén fontolja meg a `max_iter` érték növelését)
- használja a __model2__ változónevet ehhez a modellhez
- állítson be egy random_state értéket, a kísérlet megismételhetősége érdekében

In [15]:
# TODO
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC

# model2 = SVC(kernel="linear", random_state=42, max_iter=10)
model2 = LogisticRegression(random_state=42, max_iter=1000)
model2.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

### 9. LÉPÉS: Második modell kiértékelése

- értékelje ki a második modell teljesítményét a teszt halmazon a következő metrikákkal:
    - pontosság (accuracy)
    - precizitás (precision)
    - szenzitivitás (recall)
    - igazságmátrix (confusion matrix)

In [16]:
# TODO
y_pred = model2.predict(X_test)
print(classification_report(y_pred, y_test))
print(confusion_matrix(y_pred, y_test))

              precision    recall  f1-score   support

           0       0.99      0.86      0.92      1501
           1       0.25      0.81      0.38        81

    accuracy                           0.86      1582
   macro avg       0.62      0.84      0.65      1582
weighted avg       0.95      0.86      0.89      1582

[[1298  203]
 [  15   66]]


### 10. LÉPÉS: Modellek összehasonlítása

- melyik modell a megfelelőbb: model1  vagy model2,
- ha az a legfontosabb szempont, hogy ha a modell magánszemély eladót prediktál (0 azonosítójú osztály), akkor jó eséllyel ténylegesen is magánszemély legyen az eladó
- szövegesen indokolja (a vastagított részek kiválasztásával, a konkrét értékek megadása szükséges):
Az eredények alapján a __model1 | model2__ megfelelőbb, mert a __pontosság | precizitás | szenzitivitás | f1__ értéke magasabb (__x.xx > y.yy__).

Az eredények alapján a __model2__ megfelelőbb, 
mert a __precizitás__ értéke (a 0-ás osztályra vonatkozóan) magasabb
(__0.99 > 0.93__).
